In [0]:
INSERT INTO cfpb_risk.app.issue_events (
  event_id,
  issue_id,
  alert_id,
  event_ts,
  event_type,
  event_user,
  old_status,
  new_status,
  old_priority,
  new_priority,
  old_owner,
  new_owner,
  old_due_date,
  new_due_date,
  note_text,
  event_comment
)
SELECT
  sha2(concat_ws('|', i.issue_id, 'STATUS_CHANGED', cast(current_timestamp() as string)), 256) AS event_id,
  i.issue_id,
  i.alert_id,
  current_timestamp() AS event_ts,
  'STATUS_CHANGED' AS event_type,
  :p_event_user AS event_user,
  coalesce(trim(i.current_status), '') AS old_status,
  :p_new_status AS new_status,
  NULL AS old_priority,
  NULL AS new_priority,
  NULL AS old_owner,
  NULL AS new_owner,
  NULL AS old_due_date,
  NULL AS new_due_date,
  NULL AS note_text,
  CASE
    WHEN i.current_status = 'New' AND :p_new_status = 'In Review' THEN 'Issue moved into active review'
    WHEN i.current_status = 'In Review' and :p_new_status = 'Escalated' THEN 'Issue escalated for additional attention'
    WHEN :p_new_status = 'Closed' THEN 'Issue closed'
    ELSE 'Issue status updated'
  END AS event_comment
FROM cfpb_risk.app.issues i
WHERE i.issue_id = :p_issue_id
AND :p_new_status IN ('New', 'In Review', 'Escalated', 'Closed')
AND coalesce(trim(current_status), '') <> :p_new_status;

UPDATE cfpb_risk.app.issues
SET
  current_status = :p_new_status,
  updated_ts = current_timestamp()
WHERE issue_id = :p_issue_id
AND :p_new_status IN ('New', 'In Review', 'Escalated', 'Closed')
AND coalesce(trim(current_status), '') <> :p_new_status;